In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR
DATA_DIR = PROJECT_DIR / "data"
MODEL_DIR = PROJECT_DIR / "models"

## 生成數學函數數據

In [3]:
x = np.linspace(-5, 5, 100)                 # x 座標
rng = np.random.default_rng(42)

In [4]:
def generate_function(label, x, range):     
    param_1 = np.nan
    param_2 = np.nan
    param_3 = np.nan
    param_4 = np.nan

    if label == "linear":                   # 1. Linear: y = ax + b
        a = range.uniform(-2, 2)
        b = range.uniform(-5, 5)
        param_1 = a
        param_2 = b
        y_true = a*x + b

    elif label == "quadratic":              # 2. Quadratic: y = ax^2 + bx + c
        a = range.uniform(-2, 2)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-2, 2)
        c = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a*x**2 + b*x + c
        
    elif label == "cubic":                  # 3. Cubic: y = ax^3 + bx^2 + cx + d
        a = range.uniform(-2, 2)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-2, 2)
        c = range.uniform(-2, 2)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a*x**3 + b*x**2 + c*x + d

    elif label == "exponential":            # 4. Exponential: y = a * exp(bx) + cx + d
        a = range.uniform(-2.0, 2.0)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-2.0, 2.0)
        if abs(b) < 0.1:
            b = 0.1 if b >= 0 else -0.1
        c = range.uniform(-2, 2)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a * np.exp(b*x) + c*x + d

    elif label == "logarithmic":            # 5. Logarithm: y = a * log(|x|+1) + bx + c
        a = range.uniform(-2, 2)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(-2, 2)
        c = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a * np.log(np.abs(x) + 1) + b*x + c

    elif label == "sine":                   # 6. Sine: y = a * sin(bx + c) + d
        a = range.uniform(-2.0, 2.0)
        if abs(a) < 0.2:
            a = 0.2 if a >= 0 else -0.2
        b = range.uniform(0.5, 3.0)
        c = range.uniform(-np.pi/2, np.pi/2)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a * np.sin(b*x+c) + d

    elif label == "cosine":                 # 7. Cosine: y = a * cos(bx + c) + d
        a = range.uniform(-2.0, 2.0)
        if abs(a) < 0.2:
            a = 0.2 if a >= 0 else -0.2
        b = range.uniform(0.5, 3.0)
        c = range.uniform(-np.pi/2, np.pi/2)
        d = range.uniform(-3, 3)
        param_1 = a
        param_2 = b
        param_3 = c
        param_4 = d
        y_true = a * np.cos(b*x+c) + d

    elif label == "reciprocal":             # 8. Reciprocal: y = a / (x + b) + c
        a = range.uniform(-3, 3)
        if abs(a) < 0.1:
            a = 0.1 if a >= 0 else -0.1
        b = range.uniform(5.1, 10.0)
        c = range.uniform(-2, 2)
        param_1 = a
        param_2 = b
        param_3 = c
        y_true = a/(x+b) + c
        
    else:
        raise ValueError(f"未知的label: {label}")

    # noise_sigma
    y_std = np.std(y_true)
    noise_ratio = range.uniform(0.02, 0.10)
    noise_sigma = y_std * noise_ratio
    noise_sigma = max(noise_sigma, 0.001)

    # 加入 Gaussian Noise
    noise = range.normal(loc=0, scale=noise_sigma, size=len(x))
    y = y_true + noise

    params = [param_1, param_2, param_3, param_4]
    return y, params, noise_sigma

In [5]:
labels = ["linear", "quadratic", "cubic", "exponential", 
          "logarithmic", "sine", "cosine", "reciprocal"]
dataset_rows = []
metadata_rows = []
sample_id = 1

for label in labels:
    for _ in range(500):
        y, params, noise_sigma = generate_function(label, x, rng)
        dataset_row = {"id": sample_id,
                       "label": label}
        
        for i in range(100):
            dataset_row[f"y_{i:02d}"] = y[i]
        dataset_row["noise"] = noise_sigma
        dataset_rows.append(dataset_row)
        metadata_row = {"id": sample_id,
                        "label": label,
                        "param_1": params[0],
                        "param_2": params[1],
                        "param_3": params[2],
                        "param_4": params[3],
                        "noise_sigma": noise_sigma}
        metadata_rows.append(metadata_row)

        sample_id += 1

In [6]:
# 建立DataFrame
dataset_df = pd.DataFrame(dataset_rows)

In [7]:
# 儲存檔案
dataset_df.to_csv(f'{DATA_DIR}/v1/function_dataset.csv', index=False)       

In [8]:
# 建立metadata
metadata_df = pd.DataFrame(metadata_rows)

In [9]:
# 儲存檔案
metadata_df.to_csv(f'{DATA_DIR}/v1/function_metadata.csv', index=False)     

## 查看基本資訊(EDA)

In [10]:
df_dataset = pd.read_csv(f'{DATA_DIR}/v1/function_dataset.csv')

In [11]:
print(df_dataset.shape)     # 數據外型

(4000, 103)


In [12]:
df_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Columns: 103 entries, id to noise
dtypes: float64(101), int64(1), str(1)
memory usage: 3.1 MB


In [13]:
print(df_dataset.head())                # 前5筆資料

   id   label       y_00       y_01       y_02       y_03       y_04  \
0   1  linear  -5.823807  -6.532517  -6.237960  -5.722042  -5.737194   
1   2  linear  -4.944592  -4.792387  -4.580116  -4.763844  -4.447747   
2   3  linear  -3.101322  -2.920924  -3.025236  -2.836116  -2.799595   
3   4  linear  -1.310199  -1.336168  -1.338439  -1.361884  -1.364517   
4   5  linear  11.961239  12.216088  12.153224  11.294670  11.808034   

        y_05       y_06       y_07  ...      y_91      y_92      y_93  \
0  -5.541651  -5.667930  -5.066314  ...  3.504247  3.683009  3.828953   
1  -4.380902  -4.381187  -4.383589  ...  3.088656  3.217451  3.288351   
2  -2.708347  -2.521394  -2.480784  ...  4.209317  4.021939  4.295899   
3  -1.382918  -1.367030  -1.405071  ... -2.041984 -2.036898 -2.055621   
4  11.015345  10.384167  10.323507  ... -3.236843 -4.381846 -5.044619   

       y_94      y_95      y_96      y_97      y_98      y_99     noise  
0  4.031867  4.538433  4.279250  4.539366  5.125381  4

In [14]:
print(df_dataset.columns.tolist())      # 欄位

['id', 'label', 'y_00', 'y_01', 'y_02', 'y_03', 'y_04', 'y_05', 'y_06', 'y_07', 'y_08', 'y_09', 'y_10', 'y_11', 'y_12', 'y_13', 'y_14', 'y_15', 'y_16', 'y_17', 'y_18', 'y_19', 'y_20', 'y_21', 'y_22', 'y_23', 'y_24', 'y_25', 'y_26', 'y_27', 'y_28', 'y_29', 'y_30', 'y_31', 'y_32', 'y_33', 'y_34', 'y_35', 'y_36', 'y_37', 'y_38', 'y_39', 'y_40', 'y_41', 'y_42', 'y_43', 'y_44', 'y_45', 'y_46', 'y_47', 'y_48', 'y_49', 'y_50', 'y_51', 'y_52', 'y_53', 'y_54', 'y_55', 'y_56', 'y_57', 'y_58', 'y_59', 'y_60', 'y_61', 'y_62', 'y_63', 'y_64', 'y_65', 'y_66', 'y_67', 'y_68', 'y_69', 'y_70', 'y_71', 'y_72', 'y_73', 'y_74', 'y_75', 'y_76', 'y_77', 'y_78', 'y_79', 'y_80', 'y_81', 'y_82', 'y_83', 'y_84', 'y_85', 'y_86', 'y_87', 'y_88', 'y_89', 'y_90', 'y_91', 'y_92', 'y_93', 'y_94', 'y_95', 'y_96', 'y_97', 'y_98', 'y_99', 'noise']


In [15]:
print(df_dataset.dtypes)                # 資料型態

id         int64
label        str
y_00     float64
y_01     float64
y_02     float64
          ...   
y_96     float64
y_97     float64
y_98     float64
y_99     float64
noise    float64
Length: 103, dtype: object


In [16]:
print(df_dataset.isnull().sum())        # 缺失值

id       0
label    0
y_00     0
y_01     0
y_02     0
        ..
y_96     0
y_97     0
y_98     0
y_99     0
noise    0
Length: 103, dtype: int64


In [17]:
print(df_dataset['label'].value_counts())       # 查看各函數類型數量

label
linear         500
quadratic      500
cubic          500
exponential    500
logarithmic    500
sine           500
cosine         500
reciprocal     500
Name: count, dtype: int64


In [18]:
print(df_dataset['label'].value_counts(normalize=True))   # 查看各函數類型比例

label
linear         0.125
quadratic      0.125
cubic          0.125
exponential    0.125
logarithmic    0.125
sine           0.125
cosine         0.125
reciprocal     0.125
Name: proportion, dtype: float64


In [19]:
df_metadata = pd.read_csv(f'{DATA_DIR}/v1/function_metadata.csv')

In [20]:
print(df_metadata.shape)                # metadata 數據外型

(4000, 7)


In [21]:
df_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           4000 non-null   int64  
 1   label        4000 non-null   str    
 2   param_1      4000 non-null   float64
 3   param_2      4000 non-null   float64
 4   param_3      3500 non-null   float64
 5   param_4      2000 non-null   float64
 6   noise_sigma  4000 non-null   float64
dtypes: float64(5), int64(1), str(1)
memory usage: 218.9 KB


In [22]:
print(df_metadata.head())               # 前5筆資料

   id   label   param_1   param_2  param_3  param_4  noise_sigma
0   1  linear  1.095824 -0.611216      NaN      NaN     0.283372
1   2  linear  0.867561 -0.506385      NaN      NaN     0.105685
2   3  linear  0.777050  0.811166      NaN      NaN     0.081524
3   4  linear -0.076358 -1.716388      NaN      NaN     0.013990
4   5  linear -1.771072  3.009639      NaN      NaN     0.486574


In [23]:
print(df_metadata.columns.tolist())     # 欄位

['id', 'label', 'param_1', 'param_2', 'param_3', 'param_4', 'noise_sigma']


In [24]:
print(df_metadata.dtypes)               # 資料型態

id               int64
label              str
param_1        float64
param_2        float64
param_3        float64
param_4        float64
noise_sigma    float64
dtype: object


In [25]:
print(df_metadata.isnull().sum())       # 缺失值

id                0
label             0
param_1           0
param_2           0
param_3         500
param_4        2000
noise_sigma       0
dtype: int64


In [26]:
print(df_metadata['noise_sigma'].describe())    # 查看noise分布

count    4000.000000
mean        2.322989
std        15.174162
min         0.001000
25%         0.030551
50%         0.121044
75%         0.496619
max       417.516750
Name: noise_sigma, dtype: float64


## 清理資料

In [27]:
df1 = df_dataset.drop(columns=['id'])

In [28]:
df1 = df1.drop(columns=['noise'])

In [29]:
df1.to_csv(f'{DATA_DIR}/v1/function_dataset_cleaned.csv', index=False)

In [30]:
print(df1.shape)        # 清理後數據的形狀

(4000, 101)


In [31]:
# 進行數據清理, 將遺失值用"0"填補
df_metadata.fillna(0, inplace=True)

,id,label,param_1,param_2,param_3,param_4,noise_sigma
0,1,linear,1.095824,-0.611216,0.000000,0.0,0.283372
1,2,linear,0.867561,-0.506385,0.000000,0.0,0.105685
2,3,linear,0.777050,0.811166,0.000000,0.0,0.081524
3,4,linear,-0.076358,-1.716388,0.000000,0.0,0.013990
4,5,linear,-1.771072,3.009639,0.000000,0.0,0.486574
...,...,...,...,...,...,...,...
3995,3996,reciprocal,0.340139,6.443528,1.573722,0.0,0.002182
3996,3997,reciprocal,1.316830,6.375411,-0.981416,0.0,0.013718
3997,3998,reciprocal,-0.682880,6.673115,-1.784930,0.0,0.003197
3998,3999,reciprocal,-0.525149,6.122300,-1.287712,0.0,0.004232


In [32]:
# 以清理過後的資料來蓋掉原本metadata資料
df2 = df_metadata
df2.to_csv(f'{DATA_DIR}/v1/function_metadata.csv', index=False)

In [33]:
print(df2.shape)        # 清理後metadata資料形狀

(4000, 7)
